In [12]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import mean_absolute_error
import lightgbm as lgb

# Reproducibility
SEED = 42
np.random.seed(SEED)

# Paths
TRAIN_PATH = "../data/train.csv"
TEST_PATH = "../data/test.csv"

# Columns
pollutant_columns = ['valeur_CO', 'valeur_NO2', 'valeur_O3', 'valeur_PM10', 'valeur_PM25']

# Ensure models dir exists
os.makedirs("../models", exist_ok=True)


In [13]:
df = pd.read_csv(TRAIN_PATH)
df['id'] = pd.to_datetime(df['id'])
df = df.set_index('id').sort_index()

print(df.shape)
df[pollutant_columns].head()


(40991, 5)


,valeur_CO,valeur_NO2,valeur_O3,valeur_PM10,valeur_PM25
id,,,,,
2020-01-01 00:00:00,0.718,42.9,15.7,73.1,64.4
2020-01-01 01:00:00,0.587,33.6,10.1,74.8,66.0
2020-01-01 02:00:00,NaN,29.3,5.1,51.0,44.9
2020-01-01 03:00:00,0.246,30.5,7.2,27.7,25.1
2020-01-01 04:00:00,0.204,29.3,8.3,15.3,13.6


In [14]:
# Optional simple interpolation for O3 (safe small gaps)
df['valeur_O3'] = df['valeur_O3'].interpolate(method='time')

# If you already have a better imputation block elsewhere (QOLMAT), keep that and remove this.
# Ensure there are no remaining NaNs before feature creation for lags/rollings:
# You can still keep some NaNs for targets; they will be dropped after shifting.


In [15]:
cal = df.copy()

cal['hour'] = cal.index.hour
cal['dayofweek'] = cal.index.dayofweek
cal['month'] = cal.index.month
cal['dayofyear'] = cal.index.dayofyear
cal['is_weekend'] = (cal['dayofweek'] >= 5).astype(int)

# Cyclical encodings
cal['hour_sin'] = np.sin(2*np.pi*cal['hour']/24)
cal['hour_cos'] = np.cos(2*np.pi*cal['hour']/24)
cal['dow_sin'] = np.sin(2*np.pi*cal['dayofweek']/7)
cal['dow_cos'] = np.cos(2*np.pi*cal['dayofweek']/7)
cal['month_sin'] = np.sin(2*np.pi*cal['month']/12)
cal['month_cos'] = np.cos(2*np.pi*cal['month']/12)

cal.head()


,valeur_NO2,valeur_CO,valeur_O3,valeur_PM10,valeur_PM25,hour,dayofweek,month,dayofyear,is_weekend,hour_sin,hour_cos,dow_sin,dow_cos,month_sin,month_cos
id,,,,,,,,,,,,,,,,
2020-01-01 00:00:00,42.9,0.718,15.7,73.1,64.4,0,2,1,1,0,0.000000,1.000000,0.974928,-0.222521,0.5,0.866025
2020-01-01 01:00:00,33.6,0.587,10.1,74.8,66.0,1,2,1,1,0,0.258819,0.965926,0.974928,-0.222521,0.5,0.866025
2020-01-01 02:00:00,29.3,NaN,5.1,51.0,44.9,2,2,1,1,0,0.500000,0.866025,0.974928,-0.222521,0.5,0.866025
2020-01-01 03:00:00,30.5,0.246,7.2,27.7,25.1,3,2,1,1,0,0.707107,0.707107,0.974928,-0.222521,0.5,0.866025
2020-01-01 04:00:00,29.3,0.204,8.3,15.3,13.6,4,2,1,1,0,0.866025,0.500000,0.974928,-0.222521,0.5,0.866025


In [16]:
fe = cal.copy()

# Lags
for col in pollutant_columns:
    fe[f"{col}_lag1"] = fe[col].shift(1)
    fe[f"{col}_lag24"] = fe[col].shift(24)
    fe[f"{col}_lag168"] = fe[col].shift(168)

# Rollings (trailing windows)
for col in pollutant_columns:
    fe[f"{col}_roll6h"]   = fe[col].rolling(window=6, min_periods=1).mean()
    fe[f"{col}_roll24h"]  = fe[col].rolling(window=24, min_periods=1).mean()
    fe[f"{col}_roll168h"] = fe[col].rolling(window=168, min_periods=1).mean()

# Drop earliest rows that cannot have full lags
fe = fe.dropna(subset=[f"{c}_lag168" for c in pollutant_columns])
fe = fe.dropna(subset=pollutant_columns)
fe.shape


(17496, 46)

In [17]:
split_date = "2023-07-01"

train = fe.loc[:pd.to_datetime(split_date) - pd.Timedelta(seconds=1)]
valid = fe.loc[pd.to_datetime(split_date):]

X_cols = [c for c in fe.columns if c not in pollutant_columns]

X_train = train[X_cols]
y_train = train[pollutant_columns]

X_valid = valid[X_cols]
y_valid = valid[pollutant_columns]

X_train.shape, X_valid.shape


((11048, 41), (6448, 41))

In [18]:
from lightgbm import LGBMRegressor
import lightgbm as lgb
import joblib
from sklearn.metrics import mean_absolute_error
import numpy as np

models = {}
mae_scores = {}

for col in pollutant_columns:
    model = LGBMRegressor(
        objective="regression",
        learning_rate=0.05,
        num_leaves=63,
        feature_fraction=0.8,
        bagging_fraction=0.8,
        bagging_freq=5,
        random_state=SEED,
        n_estimators=5000
    )

    model.fit(
        X_train, y_train[col],
        eval_set=[(X_valid, y_valid[col])],
        eval_metric="l1",  # 'l1' == MAE
        callbacks=[
            lgb.early_stopping(stopping_rounds=200, verbose=True),
            lgb.log_evaluation(period=200)
        ]
    )

    models[col] = model
    joblib.dump(model, f"../models/{col}_lgb.pkl")

    # Use best_iteration_ if available
    if hasattr(model, "best_iteration_") and model.best_iteration_ is not None:
        y_pred = model.predict(X_valid, num_iteration=model.best_iteration_)
    else:
        y_pred = model.predict(X_valid)

    mae = mean_absolute_error(y_valid[col], y_pred)
    mae_scores[col] = mae
    print(f"{col} MAE: {mae:.4f}")

print("Average MAE:", np.mean(list(mae_scores.values())))


[LightGBM] [Warning] bagging_freq is set=5, subsample_freq=0 will be ignored. Current value: bagging_freq=5
[LightGBM] [Warning] feature_fraction is set=0.8, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.8
[LightGBM] [Warning] bagging_fraction is set=0.8, subsample=1.0 will be ignored. Current value: bagging_fraction=0.8
[LightGBM] [Warning] bagging_freq is set=5, subsample_freq=0 will be ignored. Current value: bagging_freq=5
[LightGBM] [Warning] feature_fraction is set=0.8, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.8
[LightGBM] [Warning] bagging_fraction is set=0.8, subsample=1.0 will be ignored. Current value: bagging_fraction=0.8
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000610 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 8014
[LightGBM] [Info] Number of data points in the train set: 11048, number of used features: 41
[LightGBM] [W

In [ ]:
import pandas as pd
import numpy as np

# 1) Load test and align index
test = pd.read_csv(TEST_PATH)
test['id'] = pd.to_datetime(test['id'])
test = test.set_index('id').sort_index()

# 2) Concatenate last part of train features (fe) with empty test rows
#    so lags/rollings can be computed properly for test timestamps.
all_df = pd.concat([fe.copy(), test.copy()], axis=0, sort=False)

# 3) Recreate calendar & cyclical features on the combined frame (train+test)
def add_calendar_features(frame: pd.DataFrame) -> pd.DataFrame:
    frame['hour'] = frame.index.hour
    frame['dayofweek'] = frame.index.dayofweek
    frame['month'] = frame.index.month
    frame['dayofyear'] = frame.index.dayofyear
    frame['is_weekend'] = (frame['dayofweek'] >= 5).astype(int)

    frame['hour_sin'] = np.sin(2*np.pi*frame['hour']/24)
    frame['hour_cos'] = np.cos(2*np.pi*frame['hour']/24)
    frame['dow_sin'] = np.sin(2*np.pi*frame['dayofweek']/7)
    frame['dow_cos'] = np.cos(2*np.pi*frame['dayofweek']/7)
    frame['month_sin'] = np.sin(2*np.pi*frame['month']/12)
    frame['month_cos'] = np.cos(2*np.pi*frame['month']/12)
    return frame

all_df = add_calendar_features(all_df)

# 4) Helper to compute lag/rolling features (same definitions used in train)
def add_lag_rolling(frame: pd.DataFrame) -> pd.DataFrame:
    for col in pollutant_columns:
        frame[f"{col}_lag1"]   = frame[col].shift(1)
        frame[f"{col}_lag24"]  = frame[col].shift(24)
        frame[f"{col}_lag168"] = frame[col].shift(168)
        frame[f"{col}_roll6h"]   = frame[col].rolling(window=6,   min_periods=1).mean()
        frame[f"{col}_roll24h"]  = frame[col].rolling(window=24,  min_periods=1).mean()
        frame[f"{col}_roll168h"] = frame[col].rolling(window=168, min_periods=1).mean()
    return frame

# 5) Start from a working copy. Train rows already have filled targets.
work = all_df.copy()

# Initialize lag/rolling features across the whole frame once
work = add_lag_rolling(work)

# Features used by the model (same as in Cell 6)
X_cols = [c for c in fe.columns if c not in pollutant_columns]

# 6) Walk-forward over the test timestamps, predicting and writing back into 'work'
test_index_sorted = np.sort(test.index)

for ts in test_index_sorted:
    # Ensure lag/rolling features at timestamp 'ts' are up-to-date
    # (recompute a small trailing window ending at ts for efficiency)
    loc = work.index.get_loc(ts)
    start_loc = max(0, loc - 2000)
    window = work.iloc[start_loc:loc+1].copy()
    window = add_lag_rolling(window)
    work.iloc[start_loc:loc+1] = window

    # Build feature row for ts
    rowX = work.loc[[ts], X_cols].copy()

    # If any feature is NaN (first few test steps), fill with forward/backward values as safety net
    if rowX.isna().any(axis=None):
        rowX = rowX.fillna(method='ffill').fillna(method='bfill')

    # Predict each pollutant and write back the prediction at ts
    for col in pollutant_columns:
        model = models[col]
        if hasattr(model, "best_iteration_") and model.best_iteration_:
            yhat = model.predict(rowX, num_iteration=model.best_iteration_)[0]
        else:
            yhat = model.predict(rowX)[0]
        work.at[ts, col] = float(yhat)

# 7) Collect predictions for test rows only
test_pred = work.loc[test_index_sorted, pollutant_columns].copy()

# Sanity check
assert not test_pred.isna().any().any(), "Found NaNs in test predictions"
test_pred.head()


[LightGBM] [Warning] bagging_freq is set=5, subsample_freq=0 will be ignored. Current value: bagging_freq=5
[LightGBM] [Warning] feature_fraction is set=0.8, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.8
[LightGBM] [Warning] bagging_fraction is set=0.8, subsample=1.0 will be ignored. Current value: bagging_fraction=0.8
[LightGBM] [Warning] bagging_freq is set=5, subsample_freq=0 will be ignored. Current value: bagging_freq=5
[LightGBM] [Warning] feature_fraction is set=0.8, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.8
[LightGBM] [Warning] bagging_fraction is set=0.8, subsample=1.0 will be ignored. Current value: bagging_fraction=0.8
[LightGBM] [Warning] bagging_freq is set=5, subsample_freq=0 will be ignored. Current value: bagging_freq=5
[LightGBM] [Warning] feature_fraction is set=0.8, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.8
[LightGBM] [Warning] bagging_fraction is set=0.8, subsample=1.0 will b

,valeur_CO,valeur_NO2,valeur_O3,valeur_PM10,valeur_PM25
id,,,,,
2024-09-03 23:00:00,0.195311,21.560001,46.268995,16.348773,11.031000
2024-09-04 00:00:00,0.192274,21.826005,44.658556,15.781451,10.310342
2024-09-04 01:00:00,0.187418,20.827078,42.598880,15.512503,9.977116
2024-09-04 02:00:00,0.182853,20.133130,41.349703,15.824883,9.350986
2024-09-04 03:00:00,0.183394,20.850461,37.002886,15.660627,9.245071


In [21]:
# Ensure row order matches test index
test = pd.read_csv(TEST_PATH, parse_dates=['id']).set_index('id').sort_index()
idx = test.index

# Reindex predictions to test order (safety)
pred = test_pred.reindex(idx)

# Format id exactly as "YYYY-MM-DD HH" (no minutes/seconds)
submission = pred.copy()
submission['id'] = submission.index.strftime('%Y-%m-%d %H')

# Enforce column order as in sample submission
ordered_cols = ['id', 'valeur_NO2', 'valeur_CO', 'valeur_O3', 'valeur_PM10', 'valeur_PM25']
submission = submission[ordered_cols]

# Optional: ensure float types
for c in ordered_cols[1:]:
    submission[c] = submission[c].astype(float)

save_path = "../submission.csv"
submission.to_csv(save_path, index=False)
print(f"Saved: {save_path}")
submission.head()


Saved: ../submission.csv


,id,valeur_NO2,valeur_CO,valeur_O3,valeur_PM10,valeur_PM25
id,,,,,,
2024-09-03 23:00:00,2024-09-03 23,21.560001,0.195311,46.268995,16.348773,11.031000
2024-09-04 00:00:00,2024-09-04 00,21.826005,0.192274,44.658556,15.781451,10.310342
2024-09-04 01:00:00,2024-09-04 01,20.827078,0.187418,42.598880,15.512503,9.977116
2024-09-04 02:00:00,2024-09-04 02,20.133130,0.182853,41.349703,15.824883,9.350986
2024-09-04 03:00:00,2024-09-04 03,20.850461,0.183394,37.002886,15.660627,9.245071
